In [1]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
%cd /content/drive/MyDrive/vendor-analysis-project

/content/drive/MyDrive/vendor-analysis-project


In [3]:
!pip install selenium webdriver-manager beautifulsoup4 sqlalchemy python-dotenv apscheduler
# Install packages (Colab already has some of these)
!pip install -q selenium webdriver-manager
!pip install -q sqlalchemy psycopg2-binary
!pip install -q pyspark
!pip install -q plotly
!apt-get update
!apt-get install -y chromium-chromedriver
!pip install -q selenium webdriver-manager pandas sqlalchemy
!apt-get update
!apt-get install -y chromium-chromedriver

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 77.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.3/64.3 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 512.0/512.0 kB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 48.9 MB/s eta 0:00:00
Get:1 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:2 https://cli.github.com/packages stable InRelease
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:7 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [83.2 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,288 kB]
Get:10 http

In [16]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/config/credentials.py
"""
Credentials Manager
Store and manage login credentials securely
"""
import os
from typing import Dict
from dotenv import load_dotenv

load_dotenv()  # Load from .env

class Credentials:
    """Manage application credentials"""
    def __init__(self):
        self._credentials = {}
        self.load_from_env()

    def load_from_env(self):
        """Load credentials from environment variables"""
        email = os.getenv('Ramaalfaran82@gmail.com')
        password = os.getenv('Ramaalfaran82@gmail.com')
        if email and password:
            self._credentials['email'] = email
            self._credentials['password'] = password
        else:
            raise ValueError("Credentials not found in .env file.")

    def get_credentials(self) -> Dict[str, str]:
        """
        Get stored credentials
        Returns:
            Dictionary with email and password
        """
        if not self._credentials:
            raise ValueError("Credentials not set. Check .env file.")
        return self._credentials

    def clear_credentials(self):
        """Clear stored credentials"""
        self._credentials = {}

# Global instance
credentials_manager = Credentials()

Overwriting /content/drive/MyDrive/vendor-analysis-project/config/credentials.py


In [43]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/config/settings.py
"""
Application Settings
Configure application-wide settings
"""
# Selenium Settings
SELENIUM_CONFIG = {
    'implicit_wait': 10,
    'page_load_timeout': 30,
    'headless': True, # Set to True in Colab
}
# Scraping Settings
SCRAPING_CONFIG = {
    'max_retries': 3,
    'retry_delay': 5, # seconds
    'batch_size': 50, # vendors per batch
    'save_interval': 10, # save after every 10 vendors
}
# Database Settings (we'll use SQLite for Colab)
DATABASE_CONFIG = {
    'type': 'sqlite',
    'database': '/content/drive/MyDrive/vendor-analysis-project/data/vendors.db',
}
# File Paths
PATHS = {
    'raw_data': '/content/drive/MyDrive/vendor-analysis-project/data/raw',
    'processed_data': '/content/drive/MyDrive/vendor-analysis-project/data/processed',
    'exports': '/content/drive/MyDrive/vendor-analysis-project/data/exports',
    'logs': '/content/drive/MyDrive/vendor-analysis-project/logs',
}
LOGIN_CONFIG = {
    'email_field': '#email',
    'password_field': '#password',
    'login_button': 'button.btn.btn-info.btn-block',
}
# Target Website Settings (UPDATE THESE WITH YOUR ACTUAL URLS)
WEBSITE_CONFIG = {
    'base_url': 'https://qa.mybedrock.com',
    'login_url': 'https://qa.mybedrock.com/admin',
    'vendors_url': 'https://qa.mybedrock.com/admin/clients/vendor_master',
}
# CSS Selectors (UPDATE THESE BASED ON YOUR ACTUAL WEBSITE)
SELECTORS = {
    'login': {
        'email_field': '#email',
        'password_field': '#password',
        'login_button': 'button.btn.btn-info.btn-block',
        'success_indicator': '.dashboard',
    },
    'vendors': {
        'vendor_table': '#vendors-table',
        'vendor_rows': 'tbody tr',
        'view_button': 'a',  # Inside vendor name cell
        'next_page': 'li.paginate_button.next:not(.disabled) a',
    }
}
# Create all __init__.py files
init_files = [
    '/content/drive/MyDrive/vendor-analysis-project/scraper/__init__.py',
    '/content/drive/MyDrive/vendor-analysis-project/database/__init__.py',
    '/content/drive/MyDrive/vendor-analysis-project/processing/__init__.py',
    '/content/drive/MyDrive/vendor-analysis-project/analysis/__init__.py',
    '/content/drive/MyDrive/vendor-analysis-project/visualization/__init__.py',
]
for init_file in init_files:
    with open(init_file, 'w') as f:
        f.write('# Package initialization\n')
    print(f"✓ Created: {init_file.split('/')[-2]}/__init__.py")
print("\n✅ All __init__.py files created!")

Overwriting /content/drive/MyDrive/vendor-analysis-project/config/settings.py


In [50]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/database/models.py
from sqlalchemy import Column, Integer, String, Float, DateTime, Text, ForeignKey
from sqlalchemy.ext.declarative import declarative_base
from sqlalchemy.orm import relationship
from datetime import datetime

Base = declarative_base()

class Vendor(Base):
    __tablename__ = 'vendors'

    id = Column(Integer, primary_key=True, autoincrement=True)
    bedrock_id = Column(String(50), unique=True)
    vendor_number = Column(String(50))
    vendor_name = Column(String(200))
    primary_email = Column(String(200))
    phone = Column(String(50))
    total_spend = Column(String(50))
    status = Column(String(50))
    modified_date = Column(DateTime, nullable=True)
    payment_terms = Column(String(100))
    vendor_type = Column(String(100))
    city = Column(String(100))
    state = Column(String(50))
    registered_company_name = Column(String(200))
    duns_number = Column(String(50))
    website_url = Column(String(300))
    created_at = Column(DateTime, default=datetime.utcnow)
    updated_at = Column(DateTime, default=datetime.utcnow, onupdate=datetime.utcnow)

    # Relationships
    transactions = relationship("Transaction", back_populates="vendor")
    complaints = relationship("Complaint", back_populates="vendor")
    metrics = relationship("VendorMetric", back_populates="vendor")

    def __repr__(self):
        return f"<Vendor {self.vendor_name}>"

class Transaction(Base):
    __tablename__ = 'transactions'

    id = Column(Integer, primary_key=True, autoincrement=True)
    vendor_id = Column(Integer, ForeignKey('vendors.id'))
    invoice_number = Column(String(100))
    invoice_date = Column(DateTime)
    amount = Column(Float)
    status = Column(String(50))
    created_at = Column(DateTime, default=datetime.utcnow)

    vendor = relationship("Vendor", back_populates="transactions")

class Complaint(Base):
    __tablename__ = 'complaints'

    id = Column(Integer, primary_key=True, autoincrement=True)
    vendor_id = Column(Integer, ForeignKey('vendors.id'))
    complaint_date = Column(DateTime)
    issue_type = Column(String(100))
    description = Column(Text)
    status = Column(String(50))
    created_at = Column(DateTime, default=datetime.utcnow)

    vendor = relationship("Vendor", back_populates="complaints")

class VendorMetric(Base):
    __tablename__ = 'vendor_metrics'

    id = Column(Integer, primary_key=True, autoincrement=True)
    vendor_id = Column(Integer, ForeignKey('vendors.id'))
    metric_date = Column(DateTime)
    on_time_delivery = Column(Float)
    quality_score = Column(Float)
    compliance_score = Column(Float)
    created_at = Column(DateTime, default=datetime.utcnow)

    vendor = relationship("Vendor", back_populates="metrics")

Overwriting /content/drive/MyDrive/vendor-analysis-project/database/models.py


In [19]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/database/db_connection.py
"""
Database Connection Manager
Handle database connections and session management
"""
from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from database.models import Base
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from config.settings import DATABASE_CONFIG
class DatabaseManager:
    """Manage database connections and operations"""
    def __init__(self):
        self.engine = None
        self.Session = None
    def initialize_database(self):
        """Create database engine and tables"""
        db_path = DATABASE_CONFIG['database']
        self.engine = create_engine(f'sqlite:///{db_path}', echo=False)
        # Create all tables
        Base.metadata.create_all(self.engine)
        # Create session factory
        self.Session = sessionmaker(bind=self.engine)
        print(f"✅ Database initialized at: {db_path}")
        return self.engine
    def get_session(self):
        """Get a new database session"""
        if self.Session is None:
            self.initialize_database()
        return self.Session()
    def close_session(self, session):
        """Close a database session"""
        if session:
            session.close()
# Global instance
db_manager = DatabaseManager()
# Import and initialize database
from database.db_connection import db_manager
# Create database and tables
engine = db_manager.initialize_database()
print("\n✅ Database created successfully!")
print(f"📊 Tables created: vendors, transactions, complaints, vendor_metrics")

Overwriting /content/drive/MyDrive/vendor-analysis-project/database/db_connection.py


In [20]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/utils.py
"""
Scraper Utility Functions
Helper functions for web scraping
"""
import time
import logging
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from config.settings import SCRAPING_CONFIG
# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)
def wait_for_element(driver, by, selector, timeout=10):
    """
    Wait for an element to be present
    Args:
        driver: Selenium WebDriver instance
        by: Selenium By locator (e.g., By.CSS_SELECTOR)
        selector: Element selector
        timeout: Maximum wait time in seconds
    Returns:
        WebElement if found, None otherwise
    """
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.presence_of_element_located((by, selector))
        )
        return element
    except TimeoutException:
        logger.error(f"Timeout waiting for element: {selector}")
        return None
def wait_for_clickable(driver, by, selector, timeout=10):
    """
    Wait for an element to be clickable
    Args:
        driver: Selenium WebDriver instance
        by: Selenium By locator
        selector: Element selector
        timeout: Maximum wait time in seconds
    Returns:
        WebElement if found, None otherwise
    """
    try:
        element = WebDriverWait(driver, timeout).until(
            EC.element_to_be_clickable((by, selector))
        )
        return element
    except TimeoutException:
        logger.error(f"Timeout waiting for clickable element: {selector}")
        return None
def safe_find_element(driver, by, selector):
    """Safely find an element without throwing exception
    Args:
        driver: Selenium WebDriver instance
        by: Selenium By locator
        selector: Element selector
    Returns:
        WebElement if found, None otherwise
    """
    try:
        return driver.find_element(by, selector)
    except NoSuchElementException:
        logger.warning(f"Element not found: {selector}")
        return None
def safe_click(driver, element, max_retries=3):
    """
    Safely click an element with retry logic
    Args:
        driver: Selenium WebDriver instance
        element: WebElement to click
        max_retries: Maximum number of retry attempts
    Returns:
        True if successful, False otherwise
    """
    for attempt in range(max_retries):
        try:
            element.click()
            return True
        except Exception as e:
            logger.warning(f"Click attempt {attempt + 1} failed: {str(e)}")
            time.sleep(1)
            if attempt == max_retries - 1:
                logger.error("All click attempts failed")
                return False
    return False
def extract_text(element, default=""):
    """
    Safely extract text from an element
    Args:
        element: WebElement
        default: Default value if extraction fails
    Returns:
        Extracted text or default value
    """
    try:
        return element.text.strip() if element else default
    except Exception as e:
        logger.warning(f"Text extraction failed: {str(e)}")
        return default
def retry_on_failure(func, max_retries=None, delay=None):
    """
    Decorator to retry function on failure
    Args:
        func: Function to execute
        max_retries: Maximum retry attempts
        delay: Delay between retries in seconds
    Returns:
        Function result or None if all retries fail
    """
    if max_retries is None:
        max_retries = SCRAPING_CONFIG['max_retries']
    if delay is None:
        delay = SCRAPING_CONFIG['retry_delay']
    def wrapper(*args, **kwargs):
        for attempt in range(max_retries):
            try:
                return func(*args, **kwargs)
            except Exception as e:
                logger.warning(f"Attempt {attempt + 1} failed: {str(e)}")
                if attempt < max_retries - 1:
                    time.sleep(delay)
                else:
                    logger.error(f"All {max_retries} attempts failed")
                    return None
        return None
    return wrapper
def scroll_to_element(driver, element):
    """
    Scroll to make element visible
    Args:
        driver: Selenium WebDriver instance
        element: WebElement to scroll to
    """
    try:
        driver.execute_script("arguments[0].scrollIntoView(true);", element)
        time.sleep(0.5) # Brief pause after scrolling
    except Exception as e:
        logger.warning(f"Scroll failed: {str(e)}")

Overwriting /content/drive/MyDrive/vendor-analysis-project/scraper/utils.py


In [21]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/login_handler.py
"""
Login Handler
Handle authentication and session management
"""
import logging
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from config.settings import SELENIUM_CONFIG, WEBSITE_CONFIG, SELECTORS
from scraper.utils import wait_for_element, wait_for_clickable, safe_find_element
from dotenv import load_dotenv
import os

load_dotenv()

# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
class LoginHandler:
    """Handle website authentication"""
    def __init__(self):
        self.driver = None
        self.is_logged_in = False
    def initialize_driver(self):
        """Initialize Selenium WebDriver with Chrome"""
        try:
            chrome_options = Options()
            # Colab-specific settings
            chrome_options.add_argument('--headless')
            chrome_options.add_argument('--no-sandbox')
            chrome_options.add_argument('--disable-dev-shm-usage')
            chrome_options.add_argument('--disable-gpu')
            chrome_options.add_argument('--window-size=1920,1080')
            chrome_options.add_argument('--disable-blink-features=AutomationControlled')
            chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
            chrome_options.add_experimental_option('useAutomationExtension', False)
            # Add user agent
            chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36')
            self.driver = webdriver.Chrome(options=chrome_options)
            self.driver.implicitly_wait(SELENIUM_CONFIG['implicit_wait'])
            self.driver.set_page_load_timeout(SELENIUM_CONFIG['page_load_timeout'])
            logger.info("✅ WebDriver initialized successfully")
            return self.driver
        except Exception as e:
            logger.error(f"❌ Failed to initialize WebDriver: {str(e)}")
            raise
    def login(self):
        """
        Login to the website using .env credentials
        Returns:
            bool: True if login successful, False otherwise
        """
        email = os.getenv('Ramaalfaran82@gmail.com')
        password = os.getenv('Ramaalfaran82@gmail.com')
        try:
            if not self.driver:
                self.initialize_driver()
            logger.info(f"🔐 Attempting login to: {WEBSITE_CONFIG['login_url']}")
            # Navigate to login page
            self.driver.get(WEBSITE_CONFIG['login_url'])
            time.sleep(2) # Wait for page load
            # Wait for and fill email field
            email_field = wait_for_element(
                self.driver,
                By.CSS_SELECTOR,
                SELECTORS['login']['email_field']
            )
            if not email_field:
                logger.error("❌ Email field not found")
                return False
            email_field.clear()
            email_field.send_keys(email)
            logger.info("✓ Email entered")
            # Wait for and fill password field
            password_field = wait_for_element(
                self.driver,
                By.CSS_SELECTOR,
                SELECTORS['login']['password_field']
            )
            if not password_field:
                logger.error("❌ Password field not found")
                return False
            password_field.clear()
            password_field.send_keys(password)
            logger.info("✓ Password entered")
            # Click login button
            login_button = wait_for_clickable(
                self.driver,
                By.CSS_SELECTOR,
                SELECTORS['login']['login_button']
            )
            if not login_button:
                logger.error("❌ Login button not found")
                return False
            login_button.click()
            logger.info("✓ Login button clicked")
            # Wait for successful login (check for dashboard or success indicator)
            time.sleep(3)
            # Verify login success
            success_indicator = wait_for_element(
                self.driver,
                By.CSS_SELECTOR,
                SELECTORS['login']['success_indicator'],
                timeout=10
            )
            if success_indicator:
                self.is_logged_in = True
                logger.info("✅ Login successful!")
                return True
            else:
                logger.error("❌ Login failed - success indicator not found")
                return False
        except Exception as e:
            logger.error(f"❌ Login error: {str(e)}")
            return False
    def check_login_status(self):
        """Check if currently logged in"""
        return self.is_logged_in
    def navigate_to_vendors_page(self):
        """Navigate to vendors master page"""
        try:
            if not self.is_logged_in:
                logger.error("❌ Cannot navigate - not logged in")
                return False
            logger.info(f"📄 Navigating to vendors page: {WEBSITE_CONFIG['vendors_url']}")
            self.driver.get(WEBSITE_CONFIG['vendors_url'])
            time.sleep(2)
            # Verify vendors page loaded
            vendors_table = wait_for_element(
                self.driver,
                By.CSS_SELECTOR,
                SELECTORS['vendors']['vendor_table'],
                timeout=10
            )
            if vendors_table:
                logger.info("✅ Vendors page loaded successfully")
                return True
            else:
                logger.error("❌ Vendors table not found")
                return False
        except Exception as e:
            logger.error(f"❌ Navigation error: {str(e)}")
            return False
    def close(self):
        """Close the browser and cleanup"""
        if self.driver:
            self.driver.quit()
            logger.info("🔒 Browser closed")
            self.is_logged_in = False
    def __enter__(self):
        """Context manager entry"""
        self.initialize_driver()
        return self
    def __exit__(self, exc_type, exc_val, exc_tb):
        """Context manager exit"""
        self.close()

Overwriting /content/drive/MyDrive/vendor-analysis-project/scraper/login_handler.py


In [23]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/vendor_scraper.py
"""
Vendor Scraper
Main scraping logic for extracting vendor data
"""
import logging
import time
import pandas as pd
from datetime import datetime
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from config.settings import SCRAPING_CONFIG, PATHS, SELECTORS
from scraper.login_handler import LoginHandler
from scraper.utils import wait_for_element, safe_click, extract_text, scroll_to_element
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
load_dotenv()
# Setup logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)
class VendorScraper:
    """Scrape vendor data from the website"""
    def __init__(self):
        self.login_handler = LoginHandler()
        self.vendors_data = []
        self.transactions_data = []
        self.complaints_data = []
        self.scraped_count = 0
    def initialize(self):
        """Initialize scraper and login"""
        try:
            # Initialize driver
            self.login_handler.initialize_driver()
            # Login
            success = self.login_handler.login()
            if success:
                logger.info("✅ Scraper initialized and logged in")
                return True
            else:
                logger.error("❌ Failed to login")
                return False
        except Exception as e:
            logger.error(f"❌ Initialization error: {str(e)}")
            return False
    def get_vendor_rows(self):
        """Get all vendor rows from the table"""
        try:
            # Navigate to vendors page
            if not self.login_handler.navigate_to_vendors_page():
                return []
            # Find all vendor rows
            vendor_rows = self.login_handler.driver.find_elements(
                By.CSS_SELECTOR,
                SELECTORS['vendors']['vendor_rows']
            )
            logger.info(f"📊 Found {len(vendor_rows)} vendor rows")
            return vendor_rows
        except Exception as e:
            logger.error(f"❌ Error getting vendor rows: {str(e)}")
            return []
    def extract_vendor_details(self, vendor_row, index):
        """
        Extract vendor details from a row
        Args:
            vendor_row: WebElement representing vendor row
            index: Row index
        Returns:
            dict: Vendor data
        """
        try:
            vendor_data = {
                'vendor_code': '',
                'vendor_name': '',
                'email': '',
                'phone': '',
                'address': '',
                'region': '',
                'registration_date': None,
                'status': '',
                'rating': None,
                'total_transactions': 0,
                'successful_transactions': 0,
                'total_complaints': 0,
                'avg_response_time': None,
                'avg_delivery_time': None,
                'scraped_at': datetime.now()
            }
            # Scroll to row
            scroll_to_element(self.login_handler.driver, vendor_row)
            time.sleep(0.5)
            # Extract data from table cells
            cells = vendor_row.find_elements(By.TAG_NAME, 'td')
            if len(cells) >= 3:
                vendor_data['vendor_code'] = extract_text(cells[0])
                vendor_data['vendor_name'] = extract_text(cells[1])
                vendor_data['region'] = extract_text(cells[2]) if len(cells) > 2 else ''
                vendor_data['status'] = extract_text(cells[3]) if len(cells) > 3 else 'Active'
            # Try to click "View" button to get more details
            view_button = vendor_row.find_element(By.CSS_SELECTOR, SELECTORS['vendors']['view_button'])
            if view_button and safe_click(self.login_handler.driver, view_button):
                time.sleep(2) # Wait for details page to load
                # Extract detailed information
                vendor_data = self.extract_detailed_info(vendor_data)
                # Go back to list
                self.login_handler.driver.back()
                time.sleep(2)
            logger.info(f"✓ Extracted vendor {index + 1}: {vendor_data['vendor_name']}")
            return vendor_data
        except StaleElementReferenceException:
            logger.warning(f"⚠ Stale element at index {index}, retrying...")
            return None
        except Exception as e:
            logger.error(f"❌ Error extracting vendor {index}: {str(e)}")
            return None
    def extract_detailed_info(self, vendor_data):
        """
        Extract detailed information from vendor detail page
        Args:
            vendor_data: Existing vendor data dict
        Returns:
            dict: Updated vendor data
        """
        try:
            driver = self.login_handler.driver
            # Example selectors - UPDATE THESE BASED ON YOUR ACTUAL WEBSITE
            selectors_map = {
                'email': '#vendor-email',
                'phone': '#vendor-phone',
                'address': '#vendor-address',
                'registration_date': '#reg-date',
                'rating': '#vendor-rating',
                'total_transactions': '#total-transactions',
                'successful_transactions': '#successful-transactions',
                'total_complaints': '#total-complaints',
                'avg_response_time': '#avg-response-time',
                'avg_delivery_time': '#avg-delivery-time',
            }
            for field, selector in selectors_map.items():
                element = safe_find_element(driver, By.CSS_SELECTOR, selector)
                if element:
                    text = extract_text(element)
                    # Parse based on field type
                    if field in ['total_transactions', 'successful_transactions', 'total_complaints']:
                        try:
                            vendor_data[field] = int(text.replace(',', ''))
                        except:
                            vendor_data[field] = 0
                    elif field in ['rating', 'avg_response_time', 'avg_delivery_time']:
                        try:
                            vendor_data[field] = float(text)
                        except:
                            vendor_data[field] = None
                    elif field == 'registration_date':
                        vendor_data[field] = text # Will parse later
                    else:
                        vendor_data[field] = text
            return vendor_data
        except Exception as e:
            logger.warning(f"⚠ Error extracting detailed info: {str(e)}")
            return vendor_data
    def scrape_all_vendors(self, max_vendors=None, sync_mode=False):
        """
        Scrape all vendors from the website
        Args:
            max_vendors: Maximum number of vendors to scrape (None for all)
            sync_mode: If True, only scrape IDs and sync with DB (add/remove)
        Returns:
            list: List of vendor data dictionaries
        """
        try:
            if not self.initialize():
                return []
            logger.info("🚀 Starting vendor scraping...")
            vendor_rows = self.get_vendor_rows()
            if not vendor_rows:
                logger.warning("⚠ No vendor rows found")
                return []
            total_to_scrape = min(len(vendor_rows), max_vendors) if max_vendors else len(vendor_rows)
            logger.info(f"📊 Will scrape {total_to_scrape} vendors")
            if sync_mode:
                # Sync mode: Extract IDs only and sync
                vendor_ids = [row.find_element(By.CSS_SELECTOR, 'td:nth-child(1)').text for row in vendor_rows]
                self.sync_with_db(vendor_ids)
                return vendor_ids  # Return IDs for verification
            else:
                # Full mode
                for index in range(total_to_scrape):
                    try:
                        # Re-fetch rows to avoid stale element issues
                        vendor_rows = self.get_vendor_rows()
                        if index >= len(vendor_rows):
                            logger.warning(f"⚠ Index {index} out of range, stopping")
                            break
                        vendor_data = self.extract_vendor_details(vendor_rows[index], index)
                        if vendor_data:
                            self.vendors_data.append(vendor_data)
                            self.scraped_count += 1
                            # Save periodically
                            if self.scraped_count % SCRAPING_CONFIG['save_interval'] == 0:
                                self.save_data_incremental()
                                logger.info(f"💾 Progress saved: {self.scraped_count} vendors")
                        # Rate limiting
                        time.sleep(1)
                    except Exception as e:
                        logger.error(f"❌ Error scraping vendor {index}: {str(e)}")
                        continue
                # Final save
                self.save_data_incremental()
                logger.info(f"✅ Scraping complete! Total vendors scraped: {self.scraped_count}")
                return self.vendors_data
        except Exception as e:
            logger.error(f"❌ Scraping error: {str(e)}")
            return self.vendors_data
        finally:
            self.cleanup()
    def sync_with_db(self, current_ids):
        """Sync scraped IDs with DB (remove deleted, add new)"""
        engine = create_engine('sqlite:////content/drive/MyDrive/vendor-analysis-project/vendors.db')
        db_df = pd.read_sql("SELECT vendor_code FROM vendors", engine)
        db_ids = set(db_df['vendor_code'])
        current_ids_set = set(current_ids)
        # Remove deleted
        deleted = db_ids - current_ids_set
        if deleted:
            conn = engine.connect()
            conn.execute("DELETE FROM vendors WHERE vendor_code IN (?)", [tuple(deleted)])
            conn.close()
            logger.info(f"Removed {len(deleted)} vendors from DB")
        # Add new (scrape details for them)
        new = current_ids_set - db_ids
        if new:
            logger.info(f"Adding {len(new)} new vendors - scraping details")
            # Here, re-scrape details for new IDs only (implement loop similar to full scrape)
            for new_id in new:
                # Navigate and scrape detail for new_id (add logic to find row by ID)
                pass  # Placeholder - add your detail scrape code
    def save_data_incremental(self):
        """Save scraped data incrementally"""
        try:
            if not self.vendors_data:
                return
            timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
            # Save vendors data
            vendors_df = pd.DataFrame(self.vendors_data)
            vendors_file = f"{PATHS['raw_data']}/vendors_raw_{timestamp}.csv"
            vendors_df.to_csv(vendors_file, index=False)
            logger.info(f"💾 Saved {len(self.vendors_data)} vendors to {vendors_file}")
        except Exception as e:
            logger.error(f"❌ Error saving data: {str(e)}")
    def cleanup(self):
        """Cleanup resources"""
        if self.login_handler:
            self.login_handler.close()
        logger.info("🧹 Cleanup complete")
def safe_find_element(driver, by, selector):
    """Helper function to safely find element"""
    try:
        return driver.find_element(by, selector)
    except NoSuchElementException:
        return None

Overwriting /content/drive/MyDrive/vendor-analysis-project/scraper/vendor_scraper.py


In [51]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/database/queries.py
from sqlalchemy.orm import Session
from database.models import Vendor, Transaction, Complaint, VendorMetric
from typing import List, Dict
import logging

logger = logging.getLogger(__名称)

class VendorQueries:
    @staticmethod
    def bulk_insert_vendors(vendors: List[Dict]):
        session = None
        try:
            session = Session(bind=db_manager.engine)
            existing = {v.bedrock_id for v in session.query(Vendor.bedrock_id).all()}
            new_vendors = [Vendor(**v) for v in vendors if v['bedrock_id'] not in existing]
            if new_vendors:
                session.bulk_save_objects(new_vendors)
                session.commit()
            return len(new_vendors)
        except Exception as e:
            if session:
                session.rollback()
            logger.error(f"Bulk insert failed: {e}")
            return 0
        finally:
            if session:
                session.close()

Overwriting /content/drive/MyDrive/vendor-analysis-project/database/queries.py


In [25]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/notebooks/01_scraping_test.ipynb
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Vendor Scraping Test\n",
    "Test the scraper functionality"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Mount Google Drive\n",
    "from google.colab import drive\n",
    "drive.mount('/content/drive')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Add project to path\n",
    "import sys\n",
    "sys.path.append('/content/drive/MyDrive/vendor-analysis-project')"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import scraper\n",
    "from scraper.vendor_scraper import VendorScraper\n",
    "from config.credentials import credentials_manager"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Credentials loaded from .env automatically"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize scraper\n",
    "scraper = VendorScraper()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test full scraping (first 5 vendors)\n",
    "vendors_data = scraper.scrape_all_vendors(max_vendors=5, sync_mode=False)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Test sync mode\n",
    "sync_ids = scraper.scrape_all_vendors(sync_mode=True)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Display results\n",
    "import pandas as pd\n",
    "df = pd.DataFrame(vendors_data)\n",
    "df.head()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}
# Test imports
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
print("Testing imports...")
try:
    from config import settings
    print("✅ Settings imported")
    from database.db_connection import db_manager
    print("✅ Database manager imported")
    from scraper.login_handler import LoginHandler
    print("✅ Login handler imported")
    from scraper.vendor_scraper import VendorScraper
    print("✅ Vendor scraper imported")
    from database.queries import VendorQueries
    print("✅ Queries imported")
    print("\n🎉 All imports successful!")
except Exception as e:
    print(f"❌ Import error: {str(e)}")
# Let's create a helper to update selectors
def update_website_config(base_url, login_url, vendors_url):
    """Update website URLs"""
    config_path = '/content/drive/MyDrive/vendor-analysis-project/config/settings.py'
    # Read current content
    with open(config_path, 'r') as f:
        content = f.read()
    # Update URLs
    content = content.replace(
        "'base_url': 'https://qa.mybedrock.com'",
        f"'base_url': '{base_url}'"
    )
    content = content.replace(
        "'login_url': 'https://qa.mybedrock.com/admin'",
        f"'login_url': '{login_url}'"
    )
    content = content.replace(
        "'vendors_url': 'https://qa.mybedrock.com/admin/clients/vendor_master'",
        f"'vendors_url': '{vendors_url}'"
    )
    # Write back
    with open(config_path, 'w') as f:
        f.write(content)
    print("✅ Website URLs updated!")
# Example usage (REPLACE WITH YOUR ACTUAL URLs):
# update_website_config(
# base_url='https://qa.mybedrock.com',
# login_url='https://qa.mybedrock.com/admin',
# vendors_url='https://qa.mybedrock.com/admin/clients/vendor_master'
# )
# Update settings.py with your website URLs
settings_path = '/content/drive/MyDrive/vendor-analysis-project/config/settings.py'
# Read the current settings
with open(settings_path, 'r') as f:
    settings_content = f.read()
# Replace URLs
settings_content = settings_content.replace(
    "'base_url': 'https://qa.mybedrock.com'",
    "'base_url': 'https://qa.mybedrock.com'"
)
settings_content = settings_content.replace(
    "'login_url': 'https://qa.mybedrock.com/admin'",
    "'login_url': 'https://qa.mybedrock.com/admin'"
)
settings_content = settings_content.replace(
    "'vendors_url': 'https://qa.mybedrock.com/admin/clients/vendor_master'",
    "'vendors_url': 'https://qa.mybedrock.com/admin/clients/vendor_master'" # We'll verify this
)
# Save updated settings
with open(settings_path, 'w') as f:
    f.write(settings_content)
print("✅ URLs updated successfully!")
print("\nUpdated configuration:")
print("- Base URL: https://qa.mybedrock.com")
print("- Login URL: https://qa.mybedrock.com/admin")
print("- Vendors URL: https://qa.mybedrock.com/admin/clients/vendor_master")

Overwriting /content/drive/MyDrive/vendor-analysis-project/notebooks/01_scraping_test.ipynb


In [32]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/selector_finder.py
"""
Selector Finder Tool
Automatically detects login selectors from Bedrock QA login page
and updates settings.py + tests login
"""
import logging
import os
import sys
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from dotenv import load_dotenv

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Add project root to path
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')

# Load environment variables
load_dotenv()
EMAIL = os.getenv('BEDROCK_EMAIL')
PASSWORD = os.getenv('BEDROCK_PASSWORD')

if not EMAIL or not PASSWORD:
    raise RuntimeError(
        "\nBEDROCK_EMAIL and/or BEDROCK_PASSWORD missing in .env file!\n"
        "Add to /content/drive/MyDrive/vendor-analysis-project/.env:\n"
        "BEDROCK_EMAIL=Ramaalfaran82@gmail.com\n"
        "BEDROCK_PASSWORD=Ramaalfaran82@gmail.com"
    )

class SelectorFinder:
    """Tool to detect and verify login selectors"""
    def __init__(self, url):
        self.url = url
        self.driver = None
        self.wait = None

    def initialize_driver(self):
        """Initialize Chrome driver in headless mode"""
        chrome_options = Options()
        chrome_options.add_argument('--headless')
        chrome_options.add_argument('--no-sandbox')
        chrome_options.add_argument('--disable-dev-shm-usage')
        chrome_options.add_argument('--window-size=1920,1080')
        chrome_options.add_argument('--disable-blink-features=AutomationControlled')
        chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
        self.driver = webdriver.Chrome(options=chrome_options)
        self.wait = WebDriverWait(self.driver, 15)
        logger.info("Driver initialized")

    def load_page(self):
        """Load the login page"""
        self.driver.get(self.url)
        self.wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        time.sleep(2)
        logger.info(f"Loaded: {self.url}")

    def find_login_elements(self):
        """Detect email, password, and login button"""
        print("\nSEARCHING FOR LOGIN ELEMENTS...")
        print("=" * 70)

        found = {}

        # --- EMAIL FIELD ---
        print("\nEMAIL FIELD:")
        email_selectors = [
            ('#email', By.ID, 'email'),
            ('input[name="email"]', By.NAME, 'email'),
            ('input[type="email"]', By.CSS_SELECTOR, 'input[type="email"]'),
        ]
        for label, by, sel in email_selectors:
            try:
                elem = self.driver.find_element(by, sel)
                print(f"   FOUND: {label}")
                found['email'] = f"#{sel}" if by == By.ID else sel
                break
            except:
                continue
        else:
            print("   NOT FOUND")
            found['email'] = '#email'  # fallback

        # --- PASSWORD FIELD ---
        print("\nPASSWORD FIELD:")
        pwd_selectors = [
            ('#password', By.ID, 'password'),
            ('input[name="password"]', By.NAME, 'password'),
            ('input[type="password"]', By.CSS_SELECTOR, 'input[type="password"]'),
        ]
        for label, by, sel in pwd_selectors:
            try:
                elem = self.driver.find_element(by, sel)
                print(f"   FOUND: {label}")
                found['password'] = f"#{sel}" if by == By.ID else sel
                break
            except:
                continue
        else:
            print("   NOT FOUND")
            found['password'] = '#password'

        # --- LOGIN BUTTON ---
        print("\nLOGIN BUTTON:")
        btn_selectors = [
            ('button.btn.btn-info.btn-block', By.CSS_SELECTOR, 'button.btn.btn-info.btn-block'),
            ('button[type="submit"]', By.CSS_SELECTOR, 'button[type="submit"]'),
            ('//button[contains(text(), "Login")]', By.XPATH, '//button[contains(text(), "Login")]'),
        ]
        for label, by, sel in btn_selectors:
            try:
                elem = self.driver.find_element(by, sel)
                print(f"   FOUND: {label}")
                found['button'] = sel
                break
            except:
                continue
        else:
            print("   NOT FOUND")
            found['button'] = 'button.btn.btn-info.btn-block'

        print("\n" + "=" * 70)
        return found

    def close(self):
        """Close driver"""
        if self.driver:
            self.driver.quit()

# =============================================================================
# MAIN: FIND SELECTORS → UPDATE settings.py → TEST LOGIN
# =============================================================================
def find_and_update_selectors():
    """Main function: detect selectors, update settings, test login"""
    LOGIN_URL = 'https://qa.mybedrock.com/admin'

    finder = SelectorFinder(LOGIN_URL)
    try:
        finder.initialize_driver()
        finder.load_page()
        selectors = finder.find_login_elements()

        # === UPDATE settings.py ===
        settings_path = '/content/drive/MyDrive/vendor-analysis-project/config/settings.py'
        if not os.path.exists(settings_path):
            raise FileNotFoundError(f"settings.py not found at {settings_path}")

        with open(settings_path, 'r') as f:
            content = f.read()

        # Replace old selectors
        replacements = [
            ("'email_field':.*", f"'email_field': '{selectors['email']}',"),
            ("'password_field':.*", f"'password_field': '{selectors['password']}',"),
            ("'login_button':.*", f"'login_button': '{selectors['button']}',"),
        ]

        for old, new in replacements:
            import re
            content = re.sub(old, new, content, flags=re.IGNORECASE)

        with open(settings_path, 'w') as f:
            f.write(content)

        print("\nSELECTORS UPDATED IN settings.py")
        print("-" * 50)
        print(f" Email     : {selectors['email']}")
        print(f" Password  : {selectors['password']}")
        print(f" Button    : {selectors['button']}")
        print("-" * 50)

        return selectors

    except Exception as e:
        print(f"\nERROR during selector detection: {e}")
        return None
    finally:
        finder.close()

# =============================================================================
# TEST LOGIN USING LoginHandler
# =============================================================================
def test_login_with_handler():
    """Test login using the updated LoginHandler"""
    print("\nTESTING LOGIN WITH LoginHandler...")
    print("=" * 70)

    try:
        from scraper.login_handler import LoginHandler
        handler = LoginHandler()

        handler.initialize_driver()
        print("Logging in...")

        if handler.login():
            print("\nLOGIN SUCCESSFUL!")
            screenshot = '/content/drive/MyDrive/vendor-analysis-project/logs/login_success.png'
            os.makedirs(os.path.dirname(screenshot), exist_ok=True)
            handler.driver.save_screenshot(screenshot)
            print(f" Screenshot: {screenshot}")
            print(f" Current URL: {handler.driver.current_url}")
        else:
            print("\nLOGIN FAILED")
            screenshot = '/content/drive/MyDrive/vendor-analysis-project/logs/login_failed.png'
            os.makedirs(os.path.dirname(screenshot), exist_ok=True)
            handler.driver.save_screenshot(screenshot)
            print(f" Error screenshot: {screenshot}")

    except Exception as e:
        print(f"\nLOGIN TEST ERROR: {e}")
        import traceback
        traceback.print_exc()
    finally:
        handler.close()

# =============================================================================
# RUN EVERYTHING
# =============================================================================
if __name__ == "__main__":
    print("BEDROCK SELECTOR FINDER & LOGIN TEST")
    print("=" * 80)

    # Step 1: Find & update selectors
    updated = find_and_update_selectors()
    if not updated:
        print("Failed to update selectors. Exiting.")
        sys.exit(1)

    # Step 2: Test login
    test_login_with_handler()

    print("\n" + "=" * 80)
    print("SELECTOR FINDER + LOGIN TEST COMPLETED")
    print("=" * 80)

Overwriting /content/drive/MyDrive/vendor-analysis-project/scraper/selector_finder.py


In [33]:
# Run this cell **once** – it creates the .env file in the notebook folder
%%writefile /content/.env
BEDROCK_EMAIL=Ramaalfaran82@gmail.com
BEDROCK_PASSWORD=Ramaalfaran82@gmail.com

Overwriting /content/.env


In [ ]:
# Uncomment to test (after updating selectors):
# test_login()
# ============================================================================
# BEDROCK VENDOR SCRAPER - EXTRACT ALL DATA FROM TABLE + DETAIL PAGE
# ============================================================================
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
from datetime import datetime
import os
import json
from dotenv import load_dotenv
load_dotenv()
print("\n" + "=" * 100)
print("BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION")
print("=" * 100 + "\n")
# Configuration
EMAIL = os.getenv('Ramaalfaran82@gmail.com')
PASSWORD = os.getenv('Ramaalfaran82@gmail.com')
LOGIN_URL = 'https://qa.mybedrock.com/admin'
VENDOR_MASTER_URL = 'https://qa.mybedrock.com/admin/clients/vendor_master'
print("Configuration:")
print(f" Email: {EMAIL}\n")
# Initialize driver
print("Initializing WebDriver...")
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
driver = webdriver.Chrome(options=chrome_options)
driver.implicitly_wait(10)
driver.set_page_load_timeout(30)
print("✓ WebDriver initialized\n")
# Login
print("Step 1: Login")
print("-" * 100)
try:
    driver.get(LOGIN_URL)
    time.sleep(3)
    driver.find_element(By.ID, 'Ramaalfaran82@gmail.com').send_keys(EMAIL)
    driver.find_element(By.ID, 'Ramaalfaran82@gmail.com').send_keys(PASSWORD)
    driver.find_element(By.CSS_SELECTOR, 'button.btn.btn-info.btn-block').click()
    time.sleep(5)
    if 'authentication' not in driver.current_url:
        print("✓ Login successful\n")
    else:
        driver.quit()
        sys.exit(1)
except Exception as e:
    print(f"✗ Login error: {str(e)}")
    driver.quit()
    sys.exit(1)
# Navigate to vendor master
print("Step 2: Navigate to Vendor Master")
print("-" * 100)
driver.get(VENDOR_MASTER_URL)
time.sleep(3)
print("✓ Vendor Master loaded\n")
# Set page size to 100
print("Step 3: Set page size to 100")
print("-" * 100)
try:
    page_size_dropdown = driver.find_element(By.CSS_SELECTOR, 'select.form-control')
    driver.execute_script("arguments[0].value = '100';", page_size_dropdown)
    driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", page_size_dropdown)
    time.sleep(3)
    print("✓ Page size set to 100\n")
except:
    print("⚠ Could not change page size\n")
# STEP 4: EXTRACT ALL DATA FROM VENDOR MASTER TABLE
print("Step 4: Extract ALL data from Vendor Master Table")
print("-" * 100)
vendor_rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
total_vendors = len(vendor_rows)
print(f"Found {total_vendors} vendors\n")
vendors_data = []
print("Extracting data from table...")
for index, vendor_row in enumerate(vendor_rows):
    try:
        cells = vendor_row.find_elements(By.TAG_NAME, 'td')
        cell_texts = [cell.text.strip() for cell in cells]
        # Based on your screenshot, the table columns are:
        # 0: Checkbox
        # 1: Bedrock ID
        # 2: Vendor Number
        # 3: Vendor Name (with View button)
        # 4: DBA Name
        # 5: Onboarding Registration Status
        # 6: Audit Registration Status
        # 7: Primary Contact
        # 8: Primary Email
        # 9: Tax ID
        # 10: Audit Period
        # 11: Phone
        # 12: Total Spend
        # 13: Spend 2024
        # 14: Spend 2023
        # 15: Spend 2022
        # 16: Legacy Vendor Id
        # 17: Vendor Login Status
        # 18: Modified By
        # 19: Modified Date
        # 20: Modified Type
        # 21: Created By
        vendor_name_raw = cell_texts[3] if len(cell_texts) > 3 else ''
        vendor_name = vendor_name_raw.split('\n')[0].replace('View', '').replace('Details', '').replace('Log', '').strip()
        vendor_data = {
            # From vendor master table
            'bedrock_id': cell_texts[1] if len(cell_texts) > 1 else '',
            'vendor_number': cell_texts[2] if len(cell_texts) > 2 else '',
            'vendor_name': vendor_name,
            'dba_name': cell_texts[4] if len(cell_texts) > 4 else '',
            'onboarding_status': cell_texts[5] if len(cell_texts) > 5 else '',
            'audit_status': cell_texts[6] if len(cell_texts) > 6 else '',
            'primary_contact': cell_texts[7] if len(cell_texts) > 7 else '',
            'primary_email': cell_texts[8] if len(cell_texts) > 8 else '',
            'tax_id': cell_texts[9] if len(cell_texts) > 9 else '',
            'audit_period': cell_texts[10] if len(cell_texts) > 10 else '',
            'phone': cell_texts[11] if len(cell_texts) > 11 else '',
            'total_spend': cell_texts[12] if len(cell_texts) > 12 else '',
            'spend_2024': cell_texts[13] if len(cell_texts) > 13 else '',
            'spend_2023': cell_texts[14] if len(cell_texts) > 14 else '',
            'spend_2022': cell_texts[15] if len(cell_texts) > 15 else '',
            'legacy_vendor_id': cell_texts[16] if len(cell_texts) > 16 else '',
            'login_status': cell_texts[17] if len(cell_texts) > 17 else '',
            'modified_by': cell_texts[18] if len(cell_texts) > 18 else '',
            'modified_date': cell_texts[19] if len(cell_texts) > 19 else '',
            'modified_type': cell_texts[20] if len(cell_texts) > 20 else '',
            'created_by': cell_texts[21] if len(cell_texts) > 21 else '',
            # From detail page (will be filled later)
            'vendor_type': '',
            'payment_terms': '',
            'estimated_annual_spend': '',
            'one_time_vendor': '',
            'registered_company_name': '',
            'company_dba_name': '',
            'registered_address_1': '',
            'registered_address_2': '',
            'city': '',
            'county': '',
            'state': '',
            'postal_code': '',
            'country': '',
            'business_entity_type': '',
            'year_founded': '',
            'duns_number': '',
            'primary_contact_first_name': '',
            'primary_contact_last_name': '',
            'primary_contact_email': '',
            'primary_contact_phone': '',
            'website_url': '',
            'compliance_verification': '',
            'banking_verification': '',
            'scraped_at': datetime.now()
        }
        vendors_data.append(vendor_data)
        if (index + 1) % 10 == 0:
            print(f" Extracted {index + 1}/{total_vendors} vendors from table")
    except Exception as e:
        print(f"✗ Error extracting vendor {index + 1}: {str(e)}")
        continue
print(f"✓ Extracted {len(vendors_data)} vendors from table\n")
# STEP 5: NOW CLICK VIEW FOR EACH VENDOR AND GET DETAILED INFO
print("Step 5: Extract detailed data from View pages")
print("-" * 100)
for index in range(len(vendors_data)):
    try:
        print(f"\n[Vendor {index + 1}/{len(vendors_data)}] {vendors_data[index]['vendor_name']}")
        # Navigate to vendor master
        driver.get(VENDOR_MASTER_URL)
        time.sleep(2)
        # Re-fetch rows
        vendor_rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
        if index >= len(vendor_rows):
            print(f" ⚠ Index out of range, skipping")
            continue
        vendor_row = vendor_rows[index]
        # Click View button
        try:
            cells = vendor_row.find_elements(By.TAG_NAME, 'td')
            vendor_name_cell = cells[3]
            view_link = vendor_name_cell.find_element(By.TAG_NAME, 'a')
            driver.execute_script("arguments[0].click();", view_link)
            time.sleep(3)
            print(f" ✓ Detail page opened")
            # Extract detailed information from the page text
            page_text = driver.find_element(By.TAG_NAME, 'body').text
            # Payment Terms
            if 'Payment Terms :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Payment Terms :' in line:
                        vendors_data[index]['payment_terms'] = line.split('Payment Terms :')[1].strip()
                        break
            # Vendor Type
            if 'Vendor Type :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Vendor Type :' in line:
                        vendors_data[index]['vendor_type'] = line.split('Vendor Type :')[1].strip()
                        break
            # Estimated Annual Spend
            if 'Estimated Annual Spend :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Estimated Annual Spend :' in line:
                        vendors_data[index]['estimated_annual_spend'] = line.split('Estimated Annual Spend :')[1].strip()
                        break
            # One-time vendor
            if 'Is this a one-time vendor? :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Is this a one-time vendor? :' in line:
                        vendors_data[index]['one_time_vendor'] = line.split('Is this a one-time vendor? :')[1].strip()
                        break
            # Compliance Verification
            if 'Global Compliance Verification :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Global Compliance Verification :' in line:
                        vendors_data[index]['compliance_verification'] = line.split('Global Compliance Verification :')[1].strip()
                        break
            # Extract from input fields
            try:
                # Registered Company Name
                elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="registered_company_name"]')
                vendors_data[index]['registered_company_name'] = elem.get_attribute('value')
            except:
                pass
            try:
                # Address
                addr_inputs = driver.find_elements(By.CSS_SELECTOR, 'input[name*="address"]')
                if addr_inputs:
                    vendors_data[index]['registered_address_1'] = addr_inputs[0].get_attribute('value')
            except:
                pass
            try:
                # City
                city_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="city"]')
                vendors_data[index]['city'] = city_elem.get_attribute('value')
            except:
                pass
            try:
                # State
                state_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="state"], select[name*="state"]')
                vendors_data[index]['state'] = state_elem.get_attribute('value')
            except:
                pass
            try:
                # Postal Code
                postal_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="postal"]')
                vendors_data[index]['postal_code'] = postal_elem.get_attribute('value')
            except:
                pass
            try:
                # DUNS Number
                duns_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="duns"]')
                vendors_data[index]['duns_number'] = duns_elem.get_attribute('value')
            except:
                pass
            try:
                # Website URL
                url_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="website"]')
                vendors_data[index]['website_url'] = url_elem.get_attribute('value')
            except:
                pass
            # Show extracted info
            if vendors_data[index]['payment_terms']:
                print(f" Payment Terms: {vendors_data[index]['payment_terms']}")
            if vendors_data[index]['vendor_type']:
                print(f" Vendor Type: {vendors_data[index]['vendor_type']}")
            if vendors_data[index]['city']:
                print(f" City: {vendors_data[index]['city']}")
            print(f" ✓ Details extracted")
        except Exception as e:
            print(f" ⚠ Could not extract details: {str(e)}")
        # Save progress every 10 vendors
        if (index + 1) % 10 == 0:
            df_temp = pd.DataFrame(vendors_data)
            csv_dir = '/content/drive/MyDrive/vendor-analysis-project/data/raw'
            os.makedirs(csv_dir, exist_ok=True)
            csv_temp = f'{csv_dir}/vendors_progress_{index + 1}.csv'
            df_temp.to_csv(csv_temp, index=False)
            print(f"\n 💾 Progress saved: {index + 1} vendors")
    except Exception as e:
        print(f"✗ Error with vendor {index + 1}: {str(e)}")
        continue
# STEP 6: SAVE FINAL DATA
print("\n" + "=" * 100)
print("Step 6: Save Final Data")
print("-" * 100)
if vendors_data:
    df = pd.DataFrame(vendors_data)
    csv_dir = '/content/drive/MyDrive/vendor-analysis-project/data/raw'
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f'{csv_dir}/vendors_complete_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df.to_csv(csv_path, index=False)
    print(f"✓ Final data saved to: {csv_path}")
    print(f"\nTotal vendors scraped: {len(df)}")
    # Show summary
    print(f"\nData Summary:")
    print(f" Total records: {len(df)}")
    print(f" With email: {df['primary_email'].notna().sum()}")
    print(f" With phone: {df['phone'].notna().sum()}")
    print(f" With city: {df['city'].notna().sum()}")
    print(f" With payment terms: {df['payment_terms'].notna().sum()}")
    print(f" With vendor type: {df['vendor_type'].notna().sum()}")
    print(f"\nSample data (first 3 vendors):\n")
    print(df[['bedrock_id', 'vendor_name', 'primary_email', 'city', 'payment_terms']].head(3).to_string(index=False))
else:
    print("✗ No data scraped")
# Cleanup
print("\n" + "=" * 100)
driver.quit()
print("✓ Browser closed")
print("=" * 100)
print(f"\n✅ SCRAPING COMPLETE")
print("=" * 100 + "\n")

In [35]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/bedrock_vendor_scraper.py
"""
BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION
Extracts all data from Vendor Master table + Detail pages
"""
import sys
import os
import time
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from dotenv import load_dotenv

# Add project root
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from scraper.login_handler import LoginHandler
from config.settings import LOGIN_CONFIG

# Load environment variables
load_dotenv()
EMAIL = os.getenv('BEDROCK_EMAIL')
PASSWORD = os.getenv('BEDROCK_PASSWORD')

if not EMAIL or not PASSWORD:
    raise RuntimeError(
        "\nBEDROCK_EMAIL and/or BEDROCK_PASSWORD missing!\n"
        "Add to .env file:\n"
        "BEDROCK_EMAIL=Ramaalfaran82@gmail.com\n"
        "BEDROCK_PASSWORD=Ramaalfaran82@gmail.com"
    )

# URLs
LOGIN_URL = 'https://qa.mybedrock.com/admin'
VENDOR_MASTER_URL = 'https://qa.mybedrock.com/admin/clients/vendor_master'

print("\n" + "=" * 100)
print("BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION")
print("=" * 100 + "\n")

print("Configuration:")
print(f" Email: {EMAIL}")
print(f" Login URL: {LOGIN_URL}")
print(f" Vendor Master: {VENDOR_MASTER_URL}\n")

# Initialize Login Handler
login_handler = LoginHandler()
wait = None

try:
    # Step 1: Initialize driver
    print("Step 1: Initializing WebDriver...")
    login_handler.initialize_driver()
    driver = login_handler.driver
    wait = WebDriverWait(driver, 20)
    print("WebDriver initialized\n")

    # Step 2: Login
    print("Step 2: Logging in...")
    if not login_handler.login():
        raise RuntimeError("Login failed. Check credentials or selectors.")
    print("Login successful\n")

    # Step 3: Navigate to Vendor Master
    print("Step 3: Navigating to Vendor Master...")
    driver.get(VENDOR_MASTER_URL)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody tr')))
    time.sleep(3)
    print("Vendor Master loaded\n")

    # Step 4: Set page size to 100
    print("Step 4: Setting page size to 100...")
    try:
        dropdown = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'select.form-control')))
        driver.execute_script("arguments[0].value = '100';", dropdown)
        driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", dropdown)
        time.sleep(4)
        print("Page size set to 100\n")
    except Exception as e:
        print(f"Could not set page size: {e}\n")

    # Step 5: Extract table data
    print("Step 5: Extracting data from Vendor Master Table...")
    print("-" * 100)
    vendor_rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
    total_vendors = len(vendor_rows)
    print(f"Found {total_vendors} vendors\n")

    vendors_data = []
    for index, row in enumerate(vendor_rows):
        try:
            cells = row.find_elements(By.TAG_NAME, 'td')
            if len(cells) < 4:
                continue
            cell_texts = [c.text.strip() for c in cells]

            # Clean vendor name
            name_raw = cell_texts[3] if len(cell_texts) > 3 else ''
            vendor_name = name_raw.split('\n')[0].replace('View', '').replace('Details', '').strip()

            vendor_data = {
                # Table fields
                'bedrock_id': cell_texts[1] if len(cell_texts) > 1 else '',
                'vendor_number': cell_texts[2] if len(cell_texts) > 2 else '',
                'vendor_name': vendor_name,
                'dba_name': cell_texts[4] if len(cell_texts) > 4 else '',
                'onboarding_status': cell_texts[5] if len(cell_texts) > 5 else '',
                'audit_status': cell_texts[6] if len(cell_texts) > 6 else '',
                'primary_contact': cell_texts[7] if len(cell_texts) > 7 else '',
                'primary_email': cell_texts[8] if len(cell_texts) > 8 else '',
                'tax_id': cell_texts[9] if len(cell_texts) > 9 else '',
                'audit_period': cell_texts[10] if len(cell_texts) > 10 else '',
                'phone': cell_texts[11] if len(cell_texts) > 11 else '',
                'total_spend': cell_texts[12] if len(cell_texts) > 12 else '',
                'spend_2024': cell_texts[13] if len(cell_texts) > 13 else '',
                'spend_2023': cell_texts[14] if len(cell_texts) > 14 else '',
                'spend_2022': cell_texts[15] if len(cell_texts) > 15 else '',
                'legacy_vendor_id': cell_texts[16] if len(cell_texts) > 16 else '',
                'login_status': cell_texts[17] if len(cell_texts) > 17 else '',
                'modified_by': cell_texts[18] if len(cell_texts) > 18 else '',
                'modified_date': cell_texts[19] if len(cell_texts) > 19 else '',
                'modified_type': cell_texts[20] if len(cell_texts) > 20 else '',
                'created_by': cell_texts[21] if len(cell_texts) > 21 else '',
                # Detail page fields (to be filled)
                'vendor_type': '', 'payment_terms': '', 'estimated_annual_spend': '',
                'one_time_vendor': '', 'registered_company_name': '', 'company_dba_name': '',
                'registered_address_1': '', 'registered_address_2': '', 'city': '', 'county': '',
                'state': '', 'postal_code': '', 'country': '', 'business_entity_type': '',
                'year_founded': '', 'duns_number': '', 'primary_contact_first_name': '',
                'primary_contact_last_name': '', 'primary_contact_email': '', 'primary_contact_phone': '',
                'website_url': '', 'compliance_verification': '', 'banking_verification': '',
                'scraped_at': datetime.now().isoformat()
            }
            vendors_data.append(vendor_data)

            if (index + 1) % 10 == 0:
                print(f" Extracted {index + 1}/{total_vendors} from table")

        except Exception as e:
            print(f"Error extracting row {index + 1}: {e}")
            continue

    print(f"Extracted {len(vendors_data)} vendors from table\n")

    # Step 6: Extract detail pages
    print("Step 6: Extracting detailed data from View pages...")
    print("-" * 100)

    for idx in range(len(vendors_data)):
        vendor = vendors_data[idx]
        print(f"\n[Vendor {idx + 1}/{len(vendors_data)}] {vendor['vendor_name']}")

        try:
            # Go back to master list
            driver.get(VENDOR_MASTER_URL)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody tr')))
            time.sleep(2)

            # Re-locate rows
            rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
            if idx >= len(rows):
                print(" Index out of range, skipping")
                continue

            target_row = rows[idx]
            view_link = target_row.find_element(By.CSS_SELECTOR, 'td:nth-child(4) a')
            driver.execute_script("arguments[0].click();", view_link)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
            time.sleep(3)

            # Extract text
            page_text = driver.find_element(By.TAG_NAME, 'body').text

            def extract_label(label):
                if label in page_text:
                    for line in page_text.split('\n'):
                        if label in line:
                            return line.split(label, 1)[1].strip().split('\n')[0]
                return ''

            vendor['payment_terms'] = extract_label('Payment Terms :')
            vendor['vendor_type'] = extract_label('Vendor Type :')
            vendor['estimated_annual_spend'] = extract_label('Estimated Annual Spend :')
            vendor['one_time_vendor'] = extract_label('Is this a one-time vendor? :')
            vendor['compliance_verification'] = extract_label('Global Compliance Verification :')

            # Extract input values
            selectors = {
                'registered_company_name': 'input[name*="registered_company_name"]',
                'registered_address_1': 'input[name*="address"]:nth-of-type(1)',
                'city': 'input[name*="city"]',
                'state': 'input[name*="state"], select[name*="state"]',
                'postal_code': 'input[name*="postal"]',
                'duns_number': 'input[name*="duns"]',
                'website_url': 'input[name*="website"]',
            }

            for field, sel in selectors.items():
                try:
                    elem = driver.find_element(By.CSS_SELECTOR, sel)
                    vendor[field] = elem.get_attribute('value') or ''
                except:
                    pass

            # Log key fields
            if vendor['payment_terms']:
                print(f"  Payment Terms: {vendor['payment_terms']}")
            if vendor['city']:
                print(f"  City: {vendor['city']}")
            print("  Details extracted")

            # Save progress every 10
            if (idx + 1) % 10 == 0:
                df_temp = pd.DataFrame(vendors_data)
                os.makedirs('/content/drive/MyDrive/vendor-analysis-project/data/raw', exist_ok=True)
                path = f'/content/drive/MyDrive/vendor-analysis-project/data/raw/progress_{idx + 1}.csv'
                df_temp.to_csv(path, index=False)
                print(f"  Progress saved: {idx + 1} vendors")

        except Exception as e:
            print(f"  Failed to extract details: {e}")
            driver.save_screenshot(f"/content/drive/MyDrive/vendor-analysis-project/logs/error_vendor_{idx + 1}.png")
            continue

    # Step 7: Save final data
    print("\n" + "=" * 100)
    print("Step 7: Saving Final Data")
    print("-" * 100)

    if vendors_data:
        df = pd.DataFrame(vendors_data)
        os.makedirs('/content/drive/MyDrive/vendor-analysis-project/data/raw', exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_path = f'/content/drive/MyDrive/vendor-analysis-project/data/raw/vendors_complete_{timestamp}.csv'
        df.to_csv(final_path, index=False)
        print(f"Final data saved: {final_path}")
        print(f"Total vendors: {len(df)}")

        # Summary
        print("\nData Summary:")
        print(f" With email: {df['primary_email'].notna().sum()}")
        print(f" With phone: {df['phone'].notna().sum()}")
        print(f" With city: {df['city'].notna().sum()}")
        print(f" With payment terms: {df['payment_terms'].notna().sum()}")

        print("\nSample (first 3):")
        print(df[['vendor_name', 'primary_email', 'city', 'payment_terms']].head(3).to_string(index=False))
    else:
        print("No data scraped")

except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    if 'driver' in locals():
        driver.save_screenshot("/content/drive/MyDrive/vendor-analysis-project/logs/critical_error.png")
finally:
    if 'login_handler' in locals():
        login_handler.close()
    print("\n" + "=" * 100)
    print("SCRAPING COMPLETE")
    print("=" * 100 + "\n")

Writing /content/drive/MyDrive/vendor-analysis-project/scraper/bedrock_vendor_scraper.py


In [38]:
# Uncomment to test (after updating selectors):
# test_login()
# ============================================================================
# BEDROCK VENDOR SCRAPER - EXTRACT ALL DATA FROM TABLE + DETAIL PAGE
# ============================================================================
import sys
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
from datetime import datetime
import os
import json
from dotenv import load_dotenv
load_dotenv()
print("\n" + "=" * 100)
print("BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION")
print("=" * 100 + "\n")
# Configuration
EMAIL = os.getenv('BEDROCK_EMAIL')
PASSWORD = os.getenv('BEDROCK_PASSWORD')
LOGIN_URL = 'https://qa.mybedrock.com/admin'
VENDOR_MASTER_URL = 'https://qa.mybedrock.com/admin/clients/vendor_master'
print("Configuration:")
print(f" Email: {EMAIL}\n")
if EMAIL is None or PASSWORD is None:
    print("✗ Error: BEDROCK_EMAIL or BEDROCK_PASSWORD not found in .env file!")
    sys.exit(1)
# Initialize driver
print("Initializing WebDriver...")
chrome_options = Options()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')
chrome_options.add_argument('--disable-gpu')
chrome_options.add_argument('--window-size=1920,1080')
chrome_options.add_argument('--disable-blink-features=AutomationControlled')
chrome_options.add_experimental_option("excludeSwitches", ["enable-automation"])
chrome_options.add_experimental_option('useAutomationExtension', False)
chrome_options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
driver = webdriver.Chrome(options=chrome_options)
driver.implicitly_wait(10)
driver.set_page_load_timeout(30)
print("✓ WebDriver initialized\n")
# Login
print("Step 1: Login")
print("-" * 100)
try:
    driver.get(LOGIN_URL)
    wait = WebDriverWait(driver, 20)

    email_field = wait.until(EC.element_to_be_clickable((By.ID, 'email')))
    email_field.send_keys(EMAIL)

    password_field = driver.find_element(By.ID, 'password')
    password_field.send_keys(PASSWORD)

    driver.find_element(By.CSS_SELECTOR, 'button.btn.btn-info.btn-block').click()
    time.sleep(5)
    if 'authentication' not in driver.current_url:
        print("✓ Login successful\n")
    else:
        driver.quit()
        sys.exit(1)
except Exception as e:
    print(f"✗ Login error: {str(e)}")
    driver.quit()
    sys.exit(1)
# Navigate to vendor master
print("Step 2: Navigate to Vendor Master")
print("-" * 100)
driver.get(VENDOR_MASTER_URL)
time.sleep(3)
print("✓ Vendor Master loaded\n")
# Set page size to 100
print("Step 3: Set page size to 100")
print("-" * 100)
try:
    page_size_dropdown = driver.find_element(By.CSS_SELECTOR, 'select.form-control')
    driver.execute_script("arguments[0].value = '100';", page_size_dropdown)
    driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", page_size_dropdown)
    time.sleep(3)
    print("✓ Page size set to 100\n")
except:
    print("⚠ Could not change page size\n")
# STEP 4: EXTRACT ALL DATA FROM VENDOR MASTER TABLE
print("Step 4: Extract ALL data from Vendor Master Table")
print("-" * 100)
vendor_rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
total_vendors = len(vendor_rows)
print(f"Found {total_vendors} vendors\n")
vendors_data = []
print("Extracting data from table...")
for index, vendor_row in enumerate(vendor_rows):
    try:
        cells = vendor_row.find_elements(By.TAG_NAME, 'td')
        cell_texts = [cell.text.strip() for cell in cells]
        # Based on your screenshot, the table columns are:
        # 0: Checkbox
        # 1: Bedrock ID
        # 2: Vendor Number
        # 3: Vendor Name (with View button)
        # 4: DBA Name
        # 5: Onboarding Registration Status
        # 6: Audit Registration Status
        # 7: Primary Contact
        # 8: Primary Email
        # 9: Tax ID
        # 10: Audit Period
        # 11: Phone
        # 12: Total Spend
        # 13: Spend 2024
        # 14: Spend 2023
        # 15: Spend 2022
        # 16: Legacy Vendor Id
        # 17: Vendor Login Status
        # 18: Modified By
        # 19: Modified Date
        # 20: Modified Type
        # 21: Created By
        vendor_name_raw = cell_texts[3] if len(cell_texts) > 3 else ''
        vendor_name = vendor_name_raw.split('\n')[0].replace('View', '').replace('Details', '').replace('Log', '').strip()
        vendor_data = {
            # From vendor master table
            'bedrock_id': cell_texts[1] if len(cell_texts) > 1 else '',
            'vendor_number': cell_texts[2] if len(cell_texts) > 2 else '',
            'vendor_name': vendor_name,
            'dba_name': cell_texts[4] if len(cell_texts) > 4 else '',
            'onboarding_status': cell_texts[5] if len(cell_texts) > 5 else '',
            'audit_status': cell_texts[6] if len(cell_texts) > 6 else '',
            'primary_contact': cell_texts[7] if len(cell_texts) > 7 else '',
            'primary_email': cell_texts[8] if len(cell_texts) > 8 else '',
            'tax_id': cell_texts[9] if len(cell_texts) > 9 else '',
            'audit_period': cell_texts[10] if len(cell_texts) > 10 else '',
            'phone': cell_texts[11] if len(cell_texts) > 11 else '',
            'total_spend': cell_texts[12] if len(cell_texts) > 12 else '',
            'spend_2024': cell_texts[13] if len(cell_texts) > 13 else '',
            'spend_2023': cell_texts[14] if len(cell_texts) > 14 else '',
            'spend_2022': cell_texts[15] if len(cell_texts) > 15 else '',
            'legacy_vendor_id': cell_texts[16] if len(cell_texts) > 16 else '',
            'login_status': cell_texts[17] if len(cell_texts) > 17 else '',
            'modified_by': cell_texts[18] if len(cell_texts) > 18 else '',
            'modified_date': cell_texts[19] if len(cell_texts) > 19 else '',
            'modified_type': cell_texts[20] if len(cell_texts) > 20 else '',
            'created_by': cell_texts[21] if len(cell_texts) > 21 else '',
            # From detail page (will be filled later)
            'vendor_type': '',
            'payment_terms': '',
            'estimated_annual_spend': '',
            'one_time_vendor': '',
            'registered_company_name': '',
            'company_dba_name': '',
            'registered_address_1': '',
            'registered_address_2': '',
            'city': '',
            'county': '',
            'state': '',
            'postal_code': '',
            'country': '',
            'business_entity_type': '',
            'year_founded': '',
            'duns_number': '',
            'primary_contact_first_name': '',
            'primary_contact_last_name': '',
            'primary_contact_email': '',
            'primary_contact_phone': '',
            'website_url': '',
            'compliance_verification': '',
            'banking_verification': '',
            'scraped_at': datetime.now()
        }
        vendors_data.append(vendor_data)
        if (index + 1) % 10 == 0:
            print(f" Extracted {index + 1}/{total_vendors} vendors from table")
    except Exception as e:
        print(f"✗ Error extracting vendor {index + 1}: {str(e)}")
        continue
print(f"✓ Extracted {len(vendors_data)} vendors from table\n")
# STEP 5: NOW CLICK VIEW FOR EACH VENDOR AND GET DETAILED INFO
print("Step 5: Extract detailed data from View pages")
print("-" * 100)
for index in range(len(vendors_data)):
    try:
        print(f"\n[Vendor {index + 1}/{len(vendors_data)}] {vendors_data[index]['vendor_name']}")
        # Navigate to vendor master
        driver.get(VENDOR_MASTER_URL)
        time.sleep(2)
        # Re-fetch rows
        vendor_rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
        if index >= len(vendor_rows):
            print(f" ⚠ Index out of range, skipping")
            continue
        vendor_row = vendor_rows[index]
        # Click View button
        try:
            cells = vendor_row.find_elements(By.TAG_NAME, 'td')
            vendor_name_cell = cells[3]
            view_link = vendor_name_cell.find_element(By.TAG_NAME, 'a')
            driver.execute_script("arguments[0].click();", view_link)
            time.sleep(3)
            print(f" ✓ Detail page opened")
            # Extract detailed information from the page text
            page_text = driver.find_element(By.TAG_NAME, 'body').text
            # Payment Terms
            if 'Payment Terms :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Payment Terms :' in line:
                        vendors_data[index]['payment_terms'] = line.split('Payment Terms :')[1].strip()
                        break
            # Vendor Type
            if 'Vendor Type :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Vendor Type :' in line:
                        vendors_data[index]['vendor_type'] = line.split('Vendor Type :')[1].strip()
                        break
            # Estimated Annual Spend
            if 'Estimated Annual Spend :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Estimated Annual Spend :' in line:
                        vendors_data[index]['estimated_annual_spend'] = line.split('Estimated Annual Spend :')[1].strip()
                        break
            # One-time vendor
            if 'Is this a one-time vendor? :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Is this a one-time vendor? :' in line:
                        vendors_data[index]['one_time_vendor'] = line.split('Is this a one-time vendor? :')[1].strip()
                        break
            # Compliance Verification
            if 'Global Compliance Verification :' in page_text:
                lines = page_text.split('\n')
                for i, line in enumerate(lines):
                    if 'Global Compliance Verification :' in line:
                        vendors_data[index]['compliance_verification'] = line.split('Global Compliance Verification :')[1].strip()
                        break
            # Extract from input fields
            try:
                # Registered Company Name
                elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="registered_company_name"]')
                vendors_data[index]['registered_company_name'] = elem.get_attribute('value')
            except:
                pass
            try:
                # Address
                addr_inputs = driver.find_elements(By.CSS_SELECTOR, 'input[name*="address"]')
                if addr_inputs:
                    vendors_data[index]['registered_address_1'] = addr_inputs[0].get_attribute('value')
            except:
                pass
            try:
                # City
                city_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="city"]')
                vendors_data[index]['city'] = city_elem.get_attribute('value')
            except:
                pass
            try:
                # State
                state_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="state"], select[name*="state"]')
                vendors_data[index]['state'] = state_elem.get_attribute('value')
            except:
                pass
            try:
                # Postal Code
                postal_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="postal"]')
                vendors_data[index]['postal_code'] = postal_elem.get_attribute('value')
            except:
                pass
            try:
                # DUNS Number
                duns_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="duns"]')
                vendors_data[index]['duns_number'] = duns_elem.get_attribute('value')
            except:
                pass
            try:
                # Website URL
                url_elem = driver.find_element(By.CSS_SELECTOR, 'input[name*="website"]')
                vendors_data[index]['website_url'] = url_elem.get_attribute('value')
            except:
                pass
            # Show extracted info
            if vendors_data[index]['payment_terms']:
                print(f" Payment Terms: {vendors_data[index]['payment_terms']}")
            if vendors_data[index]['vendor_type']:
                print(f" Vendor Type: {vendors_data[index]['vendor_type']}")
            if vendors_data[index]['city']:
                print(f" City: {vendors_data[index]['city']}")
            print(f" ✓ Details extracted")
        except Exception as e:
            print(f" ⚠ Could not extract details: {str(e)}")
        # Save progress every 10 vendors
        if (index + 1) % 10 == 0:
            df_temp = pd.DataFrame(vendors_data)
            csv_dir = '/content/drive/MyDrive/vendor-analysis-project/data/raw'
            os.makedirs(csv_dir, exist_ok=True)
            csv_temp = f'{csv_dir}/vendors_progress_{index + 1}.csv'
            df_temp.to_csv(csv_temp, index=False)
            print(f"\n 💾 Progress saved: {index + 1} vendors")
    except Exception as e:
        print(f"✗ Error with vendor {index + 1}: {str(e)}")
        continue
# STEP 6: SAVE FINAL DATA
print("\n" + "=" * 100)
print("Step 6: Save Final Data")
print("-" * 100)
if vendors_data:
    df = pd.DataFrame(vendors_data)
    csv_dir = '/content/drive/MyDrive/vendor-analysis-project/data/raw'
    os.makedirs(csv_dir, exist_ok=True)
    csv_path = f'{csv_dir}/vendors_complete_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    df.to_csv(csv_path, index=False)
    print(f"✓ Final data saved to: {csv_path}")
    print(f"\nTotal vendors scraped: {len(df)}")
    # Show summary
    print(f"\nData Summary:")
    print(f" Total records: {len(df)}")
    print(f" With email: {df['primary_email'].notna().sum()}")
    print(f" With phone: {df['phone'].notna().sum()}")
    print(f" With city: {df['city'].notna().sum()}")
    print(f" With payment terms: {df['payment_terms'].notna().sum()}")
    print(f" With vendor type: {df['vendor_type'].notna().sum()}")
    print(f"\nSample data (first 3 vendors):\n")
    print(df[['bedrock_id', 'vendor_name', 'primary_email', 'city', 'payment_terms']].head(3).to_string(index=False))
else:
    print("✗ No data scraped")
# Cleanup
print("\n" + "=" * 100)
driver.quit()
print("✓ Browser closed")
print("=" * 100)
print(f"\n✅ SCRAPING COMPLETE")
print("=" * 100 + "\n")
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/bedrock_vendor_scraper.py
"""
BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION
Extracts all data from Vendor Master table + Detail pages
"""
import sys
import os
import time
from datetime import datetime
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from dotenv import load_dotenv
# Add project root
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')
from scraper.login_handler import LoginHandler
from config.settings import LOGIN_CONFIG
# Load environment variables
load_dotenv()
EMAIL = os.getenv('BEDROCK_EMAIL')
PASSWORD = os.getenv('BEDROCK_PASSWORD')
if not EMAIL or not PASSWORD:
    raise RuntimeError(
        "\nBEDROCK_EMAIL and/or BEDROCK_PASSWORD missing!\n"
        "Add to .env file:\n"
        "BEDROCK_EMAIL=Ramaalfaran82@gmail.com\n"
        "BEDROCK_PASSWORD=Ramaalfaran82@gmail.com"
    )
# URLs
LOGIN_URL = 'https://qa.mybedrock.com/admin'
VENDOR_MASTER_URL = 'https://qa.mybedrock.com/admin/clients/vendor_master'
print("\n" + "=" * 100)
print("BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION")
print("=" * 100 + "\n")
print("Configuration:")
print(f" Email: {EMAIL}")
print(f" Login URL: {LOGIN_URL}")
print(f" Vendor Master: {VENDOR_MASTER_URL}\n")
# Initialize Login Handler
login_handler = LoginHandler()
wait = None
try:
    # Step 1: Initialize driver
    print("Step 1: Initializing WebDriver...")
    login_handler.initialize_driver()
    driver = login_handler.driver
    wait = WebDriverWait(driver, 20)
    print("WebDriver initialized\n")
    # Step 2: Logging in...
    print("Step 2: Logging in...")
    if not login_handler.login():
        raise RuntimeError("Login failed. Check credentials or selectors.")
    print("Login successful\n")
    # Step 3: Navigate to Vendor Master
    print("Step 3: Navigating to Vendor Master...")
    driver.get(VENDOR_MASTER_URL)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody tr')))
    time.sleep(3)
    print("Vendor Master loaded\n")
    # Step 4: Set page size to 100
    print("Step 4: Setting page size to 100...")
    try:
        dropdown = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'select.form-control')))
        driver.execute_script("arguments[0].value = '100';", dropdown)
        driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", dropdown)
        time.sleep(4)
        print("Page size set to 100\n")
    except Exception as e:
        print(f"Could not set page size: {e}\n")
    # Step 5: Extract table data
    print("Step 5: Extracting data from Vendor Master Table...")
    print("-" * 100)
    vendor_rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
    total_vendors = len(vendor_rows)
    print(f"Found {total_vendors} vendors\n")
    vendors_data = []
    for index, row in enumerate(vendor_rows):
        try:
            cells = row.find_elements(By.TAG_NAME, 'td')
            if len(cells) < 4:
                continue
            cell_texts = [c.text.strip() for c in cells]
            # Clean vendor name
            name_raw = cell_texts[3] if len(cell_texts) > 3 else ''
            vendor_name = name_raw.split('\n')[0].replace('View', '').replace('Details', '').strip()
            vendor_data = {
                # Table fields
                'bedrock_id': cell_texts[1] if len(cell_texts) > 1 else '',
                'vendor_number': cell_texts[2] if len(cell_texts) > 2 else '',
                'vendor_name': vendor_name,
                'dba_name': cell_texts[4] if len(cell_texts) > 4 else '',
                'onboarding_status': cell_texts[5] if len(cell_texts) > 5 else '',
                'audit_status': cell_texts[6] if len(cell_texts) > 6 else '',
                'primary_contact': cell_texts[7] if len(cell_texts) > 7 else '',
                'primary_email': cell_texts[8] if len(cell_texts) > 8 else '',
                'tax_id': cell_texts[9] if len(cell_texts) > 9 else '',
                'audit_period': cell_texts[10] if len(cell_texts) > 10 else '',
                'phone': cell_texts[11] if len(cell_texts) > 11 else '',
                'total_spend': cell_texts[12] if len(cell_texts) > 12 else '',
                'spend_2024': cell_texts[13] if len(cell_texts) > 13 else '',
                'spend_2023': cell_texts[14] if len(cell_texts) > 14 else '',
                'spend_2022': cell_texts[15] if len(cell_texts) > 15 else '',
                'legacy_vendor_id': cell_texts[16] if len(cell_texts) > 16 else '',
                'login_status': cell_texts[17] if len(cell_texts) > 17 else '',
                'modified_by': cell_texts[18] if len(cell_texts) > 18 else '',
                'modified_date': cell_texts[19] if len(cell_texts) > 19 else '',
                'modified_type': cell_texts[20] if len(cell_texts) > 20 else '',
                'created_by': cell_texts[21] if len(cell_texts) > 21 else '',
                # Detail page fields (to be filled)
                'vendor_type': '', 'payment_terms': '', 'estimated_annual_spend': '',
                'one_time_vendor': '', 'registered_company_name': '', 'company_dba_name': '',
                'registered_address_1': '', 'registered_address_2': '', 'city': '', 'county': '',
                'state': '', 'postal_code': '', 'country': '', 'business_entity_type': '',
                'year_founded': '', 'duns_number': '', 'primary_contact_first_name': '',
                'primary_contact_last_name': '', 'primary_contact_email': '', 'primary_contact_phone': '',
                'website_url': '', 'compliance_verification': '', 'banking_verification': '',
                'scraped_at': datetime.now().isoformat()
            }
            vendors_data.append(vendor_data)
            if (index + 1) % 10 == 0:
                print(f" Extracted {index + 1}/{total_vendors} from table")
        except Exception as e:
            print(f"Error extracting row {index + 1}: {e}")
            continue
    print(f"Extracted {len(vendors_data)} vendors from table\n")
    # Step 6: Extract detail pages
    print("Step 6: Extracting detailed data from View pages...")
    print("-" * 100)
    for idx in range(len(vendors_data)):
        vendor = vendors_data[idx]
        print(f"\n[Vendor {idx + 1}/{len(vendors_data)}] {vendor['vendor_name']}")
        try:
            # Go back to master list
            driver.get(VENDOR_MASTER_URL)
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody tr')))
            time.sleep(2)
            # Re-locate rows
            rows = driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
            if idx >= len(rows):
                print(" Index out of range, skipping")
                continue
            target_row = rows[idx]
            view_link = target_row.find_element(By.CSS_SELECTOR, 'td:nth-child(4) a')
            driver.execute_script("arguments[0].click();", view_link)
            wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
            time.sleep(3)
            # Extract text
            page_text = driver.find_element(By.TAG_NAME, 'body').text
            def extract_label(label):
                if label in page_text:
                    for line in page_text.split('\n'):
                        if label in line:
                            return line.split(label, 1)[1].strip().split('\n')[0]
                return ''
            vendor['payment_terms'] = extract_label('Payment Terms :')
            vendor['vendor_type'] = extract_label('Vendor Type :')
            vendor['estimated_annual_spend'] = extract_label('Estimated Annual Spend :')
            vendor['one_time_vendor'] = extract_label('Is this a one-time vendor? :')
            vendor['compliance_verification'] = extract_label('Global Compliance Verification :')
            # Extract input values
            selectors = {
                'registered_company_name': 'input[name*="registered_company_name"]',
                'registered_address_1': 'input[name*="address"]:nth-of-type(1)',
                'city': 'input[name*="city"]',
                'state': 'input[name*="state"], select[name*="state"]',
                'postal_code': 'input[name*="postal"]',
                'duns_number': 'input[name*="duns"]',
                'website_url': 'input[name*="website"]',
            }
            for field, sel in selectors.items():
                try:
                    elem = driver.find_element(By.CSS_SELECTOR, sel)
                    vendor[field] = elem.get_attribute('value') or ''
                except:
                    pass
            # Log key fields
            if vendor['payment_terms']:
                print(f" Payment Terms: {vendor['payment_terms']}")
            if vendor['city']:
                print(f" City: {vendor['city']}")
            print(" Details extracted")
            # Save progress every 10
            if (idx + 1) % 10 == 0:
                df_temp = pd.DataFrame(vendors_data)
                os.makedirs('/content/drive/MyDrive/vendor-analysis-project/data/raw', exist_ok=True)
                path = f'/content/drive/MyDrive/vendor-analysis-project/data/raw/progress_{idx + 1}.csv'
                df_temp.to_csv(path, index=False)
                print(f" Progress saved: {idx + 1} vendors")
        except Exception as e:
            print(f" Failed to extract details: {e}")
            driver.save_screenshot(f"/content/drive/MyDrive/vendor-analysis-project/logs/error_vendor_{idx + 1}.png")
            continue
    # Step 7: Final save
    print("\n" + "=" * 100)
    print("Step 7: Saving Final Data")
    print("-" * 100)
    if vendors_data:
        df = pd.DataFrame(vendors_data)
        os.makedirs('/content/drive/MyDrive/vendor-analysis-project/data/raw', exist_ok=True)
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        final_path = f'/content/drive/MyDrive/vendor-analysis-project/data/raw/vendors_complete_{timestamp}.csv'
        df.to_csv(final_path, index=False)
        print(f"Final data saved: {final_path}")
        print(f"Total vendors: {len(df)}")
        # Summary
        print("\nData Summary:")
        print(f" With email: {df['primary_email'].notna().sum()}")
        print(f" With phone: {df['phone'].notna().sum()}")
        print(f" With city: {df['city'].notna().sum()}")
        print(f" With payment terms: {df['payment_terms'].notna().sum()}")
        print(f" With vendor type: {df['vendor_type'].notna().sum()}")
        print("\nSample (first 3):")
        print(df[['vendor_name', 'primary_email', 'city', 'payment_terms']].head(3).to_string(index=False))
    else:
        print("No data scraped")
except Exception as e:
    print(f"\nCRITICAL ERROR: {e}")
    if driver:
        driver.save_screenshot("/content/drive/MyDrive/vendor-analysis-project/logs/critical_error.png")
finally:
    if 'login_handler' in locals():
        login_handler.close()
    print("\n" + "=" * 100)
    print("SCRAPING COMPLETE")
    print("=" * 100 + "\n")


BEDROCK VENDOR SCRAPER - COMPLETE DATA EXTRACTION

Configuration:
 Email: Ramaalfaran82@gmail.com

Initializing WebDriver...
✓ WebDriver initialized

Step 1: Login
----------------------------------------------------------------------------------------------------
✓ Login successful

Step 2: Navigate to Vendor Master
----------------------------------------------------------------------------------------------------
✓ Vendor Master loaded

Step 3: Set page size to 100
----------------------------------------------------------------------------------------------------
✓ Page size set to 100

Step 4: Extract ALL data from Vendor Master Table
----------------------------------------------------------------------------------------------------
Found 100 vendors

Extracting data from table...
 Extracted 10/100 vendors from table
 Extracted 20/100 vendors from table
 Extracted 30/100 vendors from table
 Extracted 40/100 vendors from table
 Extracted 50/100 vendors from table
 Extracted 60/10

UsageError: Line magic function `%%writefile` not found.


In [46]:
%%writefile /content/drive/MyDrive/vendor-analysis-project/scraper/real_time_scraper.py
"""
REAL-TIME DYNAMIC VENDOR SCRAPER (FIXED & ROBUST)
- Runs every 15 minutes
- Only scrapes new/changed vendors
- Saves to DB + CSV
- Smart sync using modified_date
- Stale element safe
"""
import sys
import os
import time
import logging
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import pandas as pd

# Add project root
sys.path.append('/content/drive/MyDrive/vendor-analysis-project')

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    TimeoutException, NoSuchElementException, StaleElementReferenceException
)

from dotenv import load_dotenv
from apscheduler.schedulers.background import BackgroundScheduler

# Import your modules
from config.settings import WEBSITE_CONFIG, SELECTORS
from database.db_connection import db_manager
from database.models import Vendor
from database.queries import VendorQueries

# Load environment
load_dotenv()
EMAIL = os.getenv('BEDROCK_EMAIL')
PASSWORD = os.getenv('BEDROCK_PASSWORD')

if not EMAIL or not PASSWORD:
    raise RuntimeError("BEDROCK_EMAIL and BEDROCK_PASSWORD must be in .env")

# Logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s',
    handlers=[
        logging.FileHandler("/content/drive/MyDrive/vendor-analysis-project/logs/real_time.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# URLs
LOGIN_URL = WEBSITE_CONFIG['login_url']
VENDOR_MASTER_URL = WEBSITE_CONFIG['vendors_url']

class RealTimeVendorScraper:
    def __init__(self):
        self.driver = None
        self.wait = None
        self.last_run = None
        self.scheduler = BackgroundScheduler()
        self.is_running = False

    def initialize_driver(self):
        """Initialize headless Chrome"""
        try:
            options = Options()
            options.add_argument('--headless')
            options.add_argument('--no-sandbox')
            options.add_argument('--disable-dev-shm-usage')
            options.add_argument('--disable-gpu')
            options.add_argument('--window-size=1920,1080')
            options.add_argument('--disable-blink-features=AutomationControlled')
            options.add_experimental_option("excludeSwitches", ["enable-automation"])
            options.add_experimental_option('useAutomationExtension', False)

            self.driver = webdriver.Chrome(options=options)
            self.wait = WebDriverWait(self.driver, 20)
            self.driver.implicitly_wait(10)
            logger.info("WebDriver initialized")
            return True
        except Exception as e:
            logger.error(f"Failed to initialize driver: {e}")
            return False

    def login(self) -> bool:
        """Login to Bedrock"""
        try:
            self.driver.get(LOGIN_URL)
            email_field = self.wait.until(EC.element_to_be_clickable((By.ID, 'email')))
            email_field.clear()
            email_field.send_keys(EMAIL)

            password_field = self.driver.find_element(By.ID, 'password')
            password_field.clear()
            password_field.send_keys(PASSWORD)

            login_btn = self.driver.find_element(By.CSS_SELECTOR, 'button.btn.btn-info.btn-block')
            login_btn.click()

            time.sleep(5)
            if 'authentication' not in self.driver.current_url:
                logger.info("Login successful")
                return True
            else:
                logger.error("Login failed - still on login page")
                return False
        except Exception as e:
            logger.error(f"Login error: {e}")
            return False

    def get_last_scraped_date(self) -> Optional[datetime]:
        """Get latest modified_date from DB"""
        session = db_manager.get_session()
        try:
            result = session.query(Vendor.modified_date)\
                .filter(Vendor.modified_date.isnot(None))\
                .order_by(Vendor.modified_date.desc())\
                .first()
            return result[0] if result else None
        except Exception as e:
            logger.warning(f"Could not get last scraped date: {e}")
            return None
        finally:
            db_manager.close_session(session)

    def parse_modified_date(self, date_str: str) -> Optional[datetime]:
        """Parse modified date from table (e.g., '10/25/2025 3:45 PM')"""
        if not date_str or not date_str.strip():
            return None
        date_str = date_str.strip()
        formats = ['%m/%d/%Y %I:%M %p', '%m/%d/%y %I:%M %p']
        for fmt in formats:
            try:
                return datetime.strptime(date_str, fmt)
            except:
                continue
        logger.warning(f"Could not parse date: {date_str}")
        return None

    def extract_vendor_row(self, row) -> Optional[Dict]:
        """Extract single row from vendor master table"""
        try:
            cells = row.find_elements(By.TAG_NAME, 'td')
            if len(cells) < 22:
                return None

            cell_texts = [c.text.strip() for c in cells]
            name_raw = cell_texts[3].split('\n')[0].replace('View', '').replace('Details', '').strip()

            modified_date = self.parse_modified_date(cell_texts[19])
            if not modified_date:
                return None

            return {
                'bedrock_id': cell_texts[1],
                'vendor_number': cell_texts[2],
                'vendor_name': name_raw,
                'primary_email': cell_texts[8],
                'phone': cell_texts[11],
                'modified_date': modified_date,
                'row_element': row  # Keep reference for clicking
            }
        except Exception as e:
            logger.warning(f"Error parsing row: {e}")
            return None

    def click_view_and_extract_details(self, row_element) -> Dict:
        """Click View and extract detail page"""
        details = {}
        try:
            # Re-find the View link to avoid stale
            view_link = row_element.find_element(By.CSS_SELECTOR, 'td:nth-child(4) a')
            self.driver.execute_script("arguments[0].scrollIntoView(true);", view_link)
            time.sleep(0.5)
            self.driver.execute_script("arguments[0].click();", view_link)

            self.wait.until(EC.presence_of_element_located((By.TAG_NAME, 'body')))
            time.sleep(3)

            # Extract from page text
            page_text = self.driver.find_element(By.TAG_NAME, 'body').text
            def get_label(text, label):
                if label in text:
                    for line in text.split('\n'):
                        if label in line:
                            return line.split(label, 1)[1].strip().split('\n')[0]
                return ''

            details.update({
                'payment_terms': get_label(page_text, 'Payment Terms :'),
                'vendor_type': get_label(page_text, 'Vendor Type :'),
                'city': '',
                'state': '',
                'registered_company_name': '',
                'duns_number': '',
                'website_url': ''
            })

            # Extract input fields safely
            inputs = {
                'registered_company_name': 'input[name*="registered_company_name"]',
                'city': 'input[name*="city"]',
                'state': 'input[name*="state"], select[name*="state"]',
                'duns_number': 'input[name*="duns"]',
                'website_url': 'input[name*="website"]'
            }
            for key, selector in inputs.items():
                try:
                    el = self.driver.find_element(By.CSS_SELECTOR, selector)
                    details[key] = el.get_attribute('value') or ''
                except:
                    details[key] = ''

            # Go back
            self.driver.back()
            time.sleep(2)
            return details
        except Exception as e:
            logger.warning(f"Detail extraction failed: {e}")
            try:
                self.driver.back()
                time.sleep(1)
            except:
                pass
            return {}

    def scrape_changed_vendors(self):
        """Main scrape loop - only new/modified"""
        if not self.initialize_driver():
            return
        if not self.login():
            self.driver.quit()
            return

        try:
            # Go to vendor master
            self.driver.get(VENDOR_MASTER_URL)
            self.wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, 'tbody tr')))
            time.sleep(3)

            # Set page size to 100
            try:
                dropdown = self.wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'select.form-control')))
                self.driver.execute_script("arguments[0].value = '100';", dropdown)
                self.driver.execute_script("arguments[0].dispatchEvent(new Event('change'));", dropdown)
                time.sleep(4)
            except Exception as e:
                logger.warning(f"Could not set page size: {e}")

            # Get last scraped date
            last_scraped = self.get_last_scraped_date()
            cutoff = last_scraped - timedelta(minutes=5) if last_scraped else datetime(2000, 1, 1)
            logger.info(f"Scraping vendors modified after: {cutoff}")

            changed_vendors = []
            processed_count = 0
            i = 0

            while True:
                try:
                    # Re-find all rows every loop to prevent stale references
                    rows = self.driver.find_elements(By.CSS_SELECTOR, 'tbody tr')
                    if i >= len(rows):
                        break

                    row = rows[i]
                    vendor = self.extract_vendor_row(row)
                    if not vendor:
                        i += 1
                        continue

                    if vendor['modified_date'] > cutoff:
                        logger.info(f"Scraping: {vendor['vendor_name']} ({vendor['modified_date']})")
                        details = self.click_view_and_extract_details(row)
                        vendor.update(details)
                        vendor.pop('row_element', None)
                        changed_vendors.append(vendor)
                        processed_count += 1
                        time.sleep(1)  # Be gentle

                    i += 1
                except StaleElementReferenceException:
                    logger.debug("Stale element caught - re-finding rows")
                    time.sleep(1)
                    continue
                except Exception as e:
                    logger.warning(f"Error on row {i}: {e}. Skipping.")
                    i += 1
                    time.sleep(1)

            # Save to DB
            if changed_vendors:
                count = VendorQueries.bulk_insert_vendors(changed_vendors)
                logger.info(f"Inserted {count} new/updated vendors into DB")

                # Save CSV
                df = pd.DataFrame(changed_vendors)
                os.makedirs('/content/drive/MyDrive/vendor-analysis-project/data/raw', exist_ok=True)
                path = f"/content/drive/MyDrive/vendor-analysis-project/data/raw/delta_{datetime.now():%Y%m%d_%H%M%S}.csv"
                df.to_csv(path, index=False)
                logger.info(f"Delta saved: {path}")
            else:
                logger.info("No changes detected")

            self.last_run = datetime.now()

        except Exception as e:
            logger.error(f"Scrape error: {e}")
        finally:
            try:
                self.driver.quit()
            except:
                pass

    def start_scheduler(self, interval_minutes: int = 15):
        """Start real-time scheduler"""
        if self.is_running:
            return

        self.scheduler.add_job(
            func=self.scrape_changed_vendors,
            trigger='interval',
            minutes=interval_minutes,
            id='vendor_scrape_job',
            replace_existing=True
        )
        self.scheduler.start()
        self.is_running = True
        logger.info(f"Real-time scraper started: every {interval_minutes} minutes")

    def stop_scheduler(self):
        """Stop scheduler"""
        if self.scheduler.running:
            self.scheduler.shutdown()
            self.is_running = False
            logger.info("Real-time scraper stopped")

    def run_once(self):
        """Run immediately (for testing)"""
        logger.info("Running one-time scrape...")
        self.scrape_changed_vendors()

# =============================================================================
# MAIN: Start Real-Time Scraper
# =============================================================================
if __name__ == "__main__":
    scraper = RealTimeVendorScraper()

    # Run once immediately
    scraper.run_once()

    # Then start scheduler
    scraper.start_scheduler(interval_minutes=15)

    logger.info("REAL-TIME SCRAPER IS NOW RUNNING")
    logger.info("Press Ctrl+C to stop")

    try:
        while True:
            time.sleep(60)
    except KeyboardInterrupt:
        scraper.stop_scheduler()
        logger.info("Goodbye!")

Overwriting /content/drive/MyDrive/vendor-analysis-project/scraper/real_time_scraper.py


In [53]:
!mkdir -p /content/drive/MyDrive/vendor-analysis-project/logs
!mkdir -p /content/drive/MyDrive/vendor-analysis-project/data/raw

In [54]:
%cd /content/drive/MyDrive/vendor-analysis-project
!python scraper/real_time_scraper.py

/content/drive/MyDrive/vendor-analysis-project
✓ Created: scraper/__init__.py
✓ Created: database/__init__.py
✓ Created: processing/__init__.py
✓ Created: analysis/__init__.py
✓ Created: visualization/__init__.py

✅ All __init__.py files created!
✅ Database initialized at: /content/drive/MyDrive/vendor-analysis-project/data/vendors.db

✅ Database created successfully!
📊 Tables created: vendors, transactions, complaints, vendor_metrics
Traceback (most recent call last):
  File "/content/drive/MyDrive/vendor-analysis-project/scraper/real_time_scraper.py", line 36, in <module>
    from database.queries import VendorQueries
  File "/content/drive/MyDrive/vendor-analysis-project/database/queries.py", line 6, in <module>
    logger = logging.getLogger(__名称)
                               ^^^^^^
NameError: name '__名称' is not defined
